# R05 · Cookbook-Referenzlabor: MATH_MAP

**NHyDB-0.27.1-Notebook-Reihe · v1 · Referenzteil**

Diese Beispiele basieren auf dem offiziellen
[NHyDB Python Cookbook](https://github.com/nhydb/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (Abschnitt 18), das als maßgebliche Referenz dient.
Code und erwartete Ausgaben sind unverändert aus dem Cookbook übernommen, die Begleittexte ins
Deutsche übersetzt; dieselben Beispiele führt auch die NHyDB-Testsuite aus.

* Jede Code-Zelle ist ein **eigenständiges Beispiel** – führe eine Zelle einzeln aus oder nutze *Run All*.
* Jedes Beispiel legt seinen eigenen temporären Datenbankordner an.
* Unter jedem Beispiel steht die im Cookbook abgedruckte Ausgabe, zum Vergleich mit deiner.
* Die Hauptreihe (`../main/`) erklärt die Ideen hinter diesen APIs Schritt für Schritt.

**Inhalt** (6 ausführbare Beispiele)

* 18. MATH_MAP: Werte per Formel platzieren

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.27.1':
    print('⚠ These examples were written for NHyDB 0.27.1; results may differ.')

nhydb 0.27.1 · file format 0.16


## 18. MATH_MAP: Werte per Formel platzieren

Ein MATH-Layout speichert einen Wert in einer Zelle, die aus Eingabewerten **berechnet** wird:
`inputs → (u, V, side)` mit ganzzahligen Ausdrücken. `V` ist eine globale
Koordinate; die Engine bildet sie automatisch auf Realms ab und besitzt diese
Realms exklusiv.

In [2]:
from nhydb.math import MathExpression, MathMap

e = MathExpression.compile("gcd(a, b) + a // 2", ("a", "b"))
print(e.evaluate(a=12, b=18))

grid = MathMap.compile("grid", ("x", "y"), u="x + 1", V="y", side="x % 2")
t = grid.target(3, 5)
print(t.u, t.V_global, t.logical_realm, t.v_local, t.side)
print(grid.target(3, 10**100 + 1).logical_realm)          # V beyond 10^100 -> next logical realm
print([t.u for t in grid.targets([(1, 1), (2, 2)])])
print(MathMap.from_dict(grid.to_dict()).fingerprint == grid.fingerprint)

from nhydb.math import DEFAULT_REALM_AUTO_LIMIT, join_global_v, normalize_side, split_global_v
print(split_global_v(10**100 + 5), join_global_v(2, 5) == 10**100 + 5)
print(normalize_side("NR"), normalize_side(False), DEFAULT_REALM_AUTO_LIMIT)

12
4 5 1 5 1
2
[2, 3]
True
(2, 5) True
1 0 4096


*Erwartete Ausgabe (aus dem Cookbook):*

```text
12
4 5 1 5 1
2
[2, 3]
True
(2, 5) True
1 0 4096
```

Ausdrücke verwenden Python-Syntax für ganze Zahlen: `+ - * // % **`, Vergleiche,
`and`/`or`/`not` und die Funktionen `abs`, `gcd`, `lcm`, `isqrt`, `min`,
`max`. Nicht erlaubt: `/`, Bit-Operatoren, Attribute, bedingte
Ausdrücke. Fehler lösen `MathSyntaxError` oder `MathDomainError` aus (beide
Unterklassen von `MathError`).

Prüfe eine Abbildung vor dem Schreiben mit `dry_run` (keine Datenbank nötig):

In [3]:
from nhydb.math import MathMap, dry_run

grid = MathMap.compile("grid", ("x", "y"), u="x + 1", V="y", side="NL")
report = dry_run(grid, [(1, 1), (1, 1), (2, 10**100 + 3), (-5, 1)])
print(report.total, report.valid, report.invalid, report.collisions, report.unique_cells)
print(report.logical_realms, report.invalid_examples)

4 3 1 1 2
(1, 2) (((-5, 1), 'u must be a positive integer'),)


*Erwartete Ausgabe (aus dem Cookbook):*

```text
4 3 1 1 2
(1, 2) (((-5, 1), 'u must be a positive integer'),)
```

MATH-Daten schreiben und lesen:

In [4]:
import os
import tempfile

import nhydb
from nhydb.math import MathMap, MathMapLayoutSpec

path = os.path.join(tempfile.mkdtemp(), "math.nhydb")
with nhydb.create(path) as db:
    grid = MathMap.compile("grid", ("x", "y"), u="x + 1", V="y", side="x % 2")
    state = db.create_math_layout(MathMapLayoutSpec("grid", grid, 64), layout_id=7, payload_type="utf8")
    print(state)

    print(db.put_math(7, (1, 7), "first"))                           # one value
    report = db.put_math_batch(7, [((x, y), f"{x},{y}") for x in (2, 3) for y in (1, 10**100 + 1)],
                               commit=True)
    print(report["cells"], report["new_realms"], report["realm_extents"])
    print(db.get_math(7, (3, 10**100 + 1)), db.get_math(7, (9, 9)))
    cell = db.get_math_cell(7, (3, 10**100 + 1))
    print(cell["_realm"], cell["_u"], cell["_lane_name"], cell["_logical_realm"])
    info = db.describe_math(7)
    print(info["frozen"], info["payload_type"], info["mapped_realms"])
    try:
        db.put_math(7, (2, 1), "again")                              # cell already used
    except nhydb.MathCellConflictError:
        print("conflict")
    print(db.put_math(7, (2, 1), "replaced", overwrite=True)["u"], db.get_math(7, (2, 1)))

MathLayoutState(layout_id=7, name='grid', frozen=False, payload_type='utf8', mapped_realms=0)
{'layout_id': 7, 'physical_realm': 1, 'u': 2, 'v': 7, 'lane': 1, 'lane_name': 'NR'}
4 1 [{'logical_start': 1, 'logical_end': 2, 'physical_start': 1, 'physical_end': 2}]
3,10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001 None
2 4 NR 2
True utf8 2
conflict
3 replaced


*Erwartete Ausgabe (aus dem Cookbook):*

```text
MathLayoutState(layout_id=7, name='grid', frozen=False, payload_type='utf8', mapped_realms=0)
{'layout_id': 7, 'physical_realm': 1, 'u': 2, 'v': 7, 'lane': 1, 'lane_name': 'NR'}
4 1 [{'logical_start': 1, 'logical_end': 2, 'physical_start': 1, 'physical_end': 2}]
3,10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001 None
2 4 NR 2
True utf8 2
conflict
3 replaced
```

Ein ganzes `put_math_batch` wird zurückgerollt, wenn eine einzige Zeile fehlschlägt. Die Kurzform
`create_math_map_layout` nimmt die Ausdrücke direkt entgegen; die Abbildung lässt sich mit
`replace_math_spec` nur **vor** dem ersten Schreiben ersetzen
(danach `MappingFrozenError`):

In [5]:
import os
import tempfile

import nhydb
from nhydb.math import MathMap, MathMapLayoutSpec

path = os.path.join(tempfile.mkdtemp(), "squares.nhydb")
with nhydb.create(path) as db:
    db.create_math_map_layout("squares", ("k",), u="k", V="k * k", side="NR",
                              payload_type="varint", layout_id=1)
    new = MathMap.compile("squares", ("k",), u="k", V="k * k + 1", side="NR")
    db.replace_math_spec(1, MathMapLayoutSpec("squares", new, 64))    # allowed: nothing written yet
    db.put_math(1, (12,), 144, commit=True)
    print(db.get_math(1, (12,)), db.get_math_cell(1, (12,))["_V_global"])
    try:
        db.replace_math_spec(1, MathMapLayoutSpec("squares", new, 64))
    except nhydb.MappingFrozenError:
        print("frozen")
    print(list(db.math_layouts))

144 145
frozen
[1]


*Erwartete Ausgabe (aus dem Cookbook):*

```text
144 145
frozen
[1]
```

NumPy-Eingabe (2-D-Array, 1-D-Array oder Liste von Spalten):

In [6]:
# requires: numpy
import os
import tempfile

import numpy as np

import nhydb
from nhydb.math import MathMap, MathMapLayoutSpec

path = os.path.join(tempfile.mkdtemp(), "mathnp.nhydb")
with nhydb.create(path) as db:
    m = MathMap.compile("g", ("x", "k"), u="x // 100 + 1", V="k * 10**100 + x % 100 + 1", side="x % 2")
    db.create_math_layout(MathMapLayoutSpec("g", m, 64), layout_id=1, payload_type="f64")
    x = np.arange(1000, dtype=np.int64)
    report = db.put_math_numpy(1, np.stack([x, x % 3], axis=1), x * 0.5, commit=True)
    print(report["cells"], db.get_math(1, (999, 0)))

1000 499.5


*Erwartete Ausgabe (aus dem Cookbook):*

```text
1000 499.5
```

Finde MATH-Zellen über Eingabe-Tupel mit dem QueryEngine-Provider
`math_layout_inputs` und begrenze die Größe von Zwischenergebnissen
(Standard 2^20 Bit):

In [7]:
import os
import tempfile

import nhydb
from nhydb.math import MathExpression, MathMap, MathMapLayoutSpec, get_math_max_intermediate_bits, set_math_max_intermediate_bits
from nhydb.query import Q, QueryEngine

path = os.path.join(tempfile.mkdtemp(), "mathq.nhydb")
with nhydb.create(path) as db:
    grid = MathMap.compile("grid", ("x", "y"), u="x + 1", V="y", side="x % 2")
    db.create_math_layout(MathMapLayoutSpec("grid", grid, 64), layout_id=1, payload_type="utf8")
    db.put_math_batch(1, [((x, y), f"{x}:{y}") for x in (1, 2) for y in (1, 2)], commit=True)
    r = QueryEngine(db).execute(Q.math("math_layout_inputs", layout_id=1, inputs=[(1, 2), (2, 2), (7, 7)]),
                                return_mode="cell")
    print([(row["N"], row["value"]) for row in r["rows"]])

print(get_math_max_intermediate_bits())
e = MathExpression.compile("2 ** x", ("x",))
old = get_math_max_intermediate_bits()
set_math_max_intermediate_bits(64)
try:
    e.evaluate(x=100)
except nhydb.MathDomainError:
    print("too large")
finally:
    set_math_max_intermediate_bits(old)

[('N18', '1:2'), ('N22', '2:2'), ('N232', None)]
1048576
too large


*Erwartete Ausgabe (aus dem Cookbook):*

```text
[('N18', '1:2'), ('N22', '2:2'), ('N232', None)]
1048576
too large
```